# 03 — Feature Engineering & Baseline Model

**Scope:** feature engineering, cleaning, preprocessing pipelines, and model
training only. Evaluation, interpretation, and business framing (metrics,
calibration, thresholds, SHAP) live in `04_model_evaluation.ipynb`, which
loads the artifacts this notebook saves.

**Product framing (drives every choice here):** Phase 3 will let a user enter
a hypothetical, **not-yet-underwritten** applicant's information and get back
a predicted probability of default. The feature set is restricted to
information genuinely available at application time — before LendingClub's
own underwriting decision. Builds on `CLAUDE.md` and notebooks `01_` and `02_`
(including its bureau-feature expansion).

## 1. Load data and scope the modeling population

Two restrictions, both justified in notebook 01:
- `issue_year >= 2013` — pre-2013 volume is a small sliver of the file, and
  several bureau-derived features (including the new ones below) are
  schema-absent before then.
- Resolved loans only (`is_resolved`) — Phase 1's modeling universe is
  resolved loans; open loans have no realized outcome yet.

Loading the feature set from notebook 02's EDA expansion — base
application-time fields plus the 7 bureau features (`delinq_2yrs`,
`inq_last_6mths`, `mort_acc`, `pub_rec`, `acc_now_delinq`,
`collections_12_mths_ex_med`, `mo_sin_old_rev_tl_op`) — in a single
`read_csv` call. A single read means no join/merge step at all, so there's
no row-count-drift risk to check the way notebook 02's two-pass load had to
(that notebook was extending an already-loaded `df`; this one starts
fresh). Also loading `int_rate` and `recoveries` here — not modeling
features, kept aside only for notebook 04's business-cost framing.

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# GarbageValueCleaner/CastToCategory live in src/preprocessing.py (not defined
# inline) so notebook 04 can unpickle these fitted pipelines in a fresh kernel --
# joblib needs the exact class to be importable from wherever it was defined.
sys.path.append(str(Path('..').resolve()))
from src.preprocessing import GarbageValueCleaner, CastToCategory

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import joblib

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
RANDOM_STATE = 42


In [2]:
DATA_PATH = '../data/accepted_2007_to_2018Q4.csv'

usecols = [
    'id', 'issue_d', 'loan_status', 'term', 'home_ownership', 'purpose', 'verification_status',
    'loan_amnt', 'dti', 'annual_inc', 'fico_range_low', 'open_acc', 'total_acc',
    'revol_util', 'emp_length', 'earliest_cr_line',
    'delinq_2yrs', 'inq_last_6mths', 'mort_acc', 'pub_rec', 'acc_now_delinq',
    'collections_12_mths_ex_med', 'mo_sin_old_rev_tl_op',
    'int_rate', 'recoveries',
]
category_cols = ['term', 'home_ownership', 'purpose', 'verification_status', 'loan_status', 'emp_length']
dtype_map = {c: 'category' for c in category_cols}

df = pd.read_csv(DATA_PATH, usecols=usecols, dtype=dtype_map, low_memory=False)

# Same trailer-row fix as notebooks 01/02: a few footer rows carry a summary
# string in `id` (not a null), and left in they'd upcast issue_year to float via NaT.
n_before_id = len(df)
df = df[pd.to_numeric(df['id'], errors='coerce').notnull()].copy()
print(f'Trailer-row drop: {n_before_id:,} -> {len(df):,} rows')

df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['issue_year'] = df['issue_d'].dt.year

n_before_year = len(df)
df = df[df['issue_year'] >= 2013].copy()
print(f'issue_year >= 2013: kept {len(df):,} / {n_before_year:,} rows ({len(df) / n_before_year:.1%})')

Trailer-row drop: 2,260,701 -> 2,260,668 rows


issue_year >= 2013: kept 2,164,766 / 2,260,668 rows (95.8%)


In [3]:
credit_policy_variants = {
    'Does not meet the credit policy. Status:Fully Paid': 'Fully Paid',
    'Does not meet the credit policy. Status:Charged Off': 'Charged Off',
}
loan_status_clean = df['loan_status'].astype(object).replace(credit_policy_variants)

resolved_statuses = ['Fully Paid', 'Charged Off', 'Default']
bad_statuses = ['Charged Off', 'Default']

df['is_resolved'] = loan_status_clean.isin(resolved_statuses)
df['target'] = np.where(df['is_resolved'], loan_status_clean.isin(bad_statuses).astype(int), np.nan)

n_before_resolved = len(df)
df = df[df['is_resolved']].copy()
df['target'] = df['target'].astype(int)

print(f'Resolved-only: kept {len(df):,} / {n_before_resolved:,} rows')
df['target'].value_counts(normalize=True)

Resolved-only: kept 1,252,197 / 2,164,766 rows


target
0   0.80
1   0.20
Name: proportion, dtype: float64

## 2. Define the (expanded) feature set

**Excluded — LendingClub's own post-underwriting risk output:** `grade`,
`sub_grade`, `int_rate`. Assigned *after* underwriting — using them would be
circular and impossible to apply to Phase 3's not-yet-underwritten applicant.

**Excluded — post-origination/performance fields:** `total_pymnt`,
`recoveries`, `last_pymnt_d`, `out_prncp`, and the rest of notebook 01's drop
list — these describe how the loan performed *after* origination, which
doesn't exist yet for a new applicant. (`recoveries` and `int_rate` are still
loaded above, but only as business-cost inputs for notebook 04, not model
features.)

**Numeric (16):** `loan_amnt`, `dti`, `annual_inc`, `fico_range_low`,
`open_acc`, `total_acc`, `revol_util`, `emp_length_numeric`,
`credit_history_years`, `delinq_2yrs`, `inq_last_6mths`, `mort_acc`,
`pub_rec`, `acc_now_delinq`, `collections_12_mths_ex_med`,
`mo_sin_old_rev_tl_op` — the last 7 are new, per notebook 02's bureau-feature
EDA.

**Categorical (4):** `term`, `home_ownership`, `purpose`,
`verification_status`.

**Phase 3 note (unchanged):** `credit_history_years` and `emp_length_numeric`
are derived here using `issue_d`, but a hypothetical applicant has no
`issue_d`. In Phase 3 these will be collected directly from the user rather
than re-derived — only the garbage-value cleaning in section 3 is pipeline
logic Phase 3 actually depends on at inference time.

In [4]:
# Same derivation as notebooks 02/03 originally. Note: .map() on a categorical
# Series returns another categorical (not numeric) -- cast to float explicitly.
emp_length_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10,
}
df['emp_length_numeric'] = df['emp_length'].map(emp_length_map).astype(float)

df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
df['credit_history_years'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365.25

In [5]:
numeric_features = [
    'loan_amnt', 'dti', 'annual_inc', 'fico_range_low', 'open_acc',
    'total_acc', 'revol_util', 'emp_length_numeric', 'credit_history_years',
    'delinq_2yrs', 'inq_last_6mths', 'mort_acc', 'pub_rec',
    'acc_now_delinq', 'collections_12_mths_ex_med', 'mo_sin_old_rev_tl_op',
]
categorical_features = ['term', 'home_ownership', 'purpose', 'verification_status']

X = df[numeric_features + categorical_features]
y = df['target']
len(numeric_features), len(categorical_features)

(16, 4)

## 3. Clean known garbage values

Five known garbage-value patterns now, per notebook 01's audit (sections
9/9b):
- `dti == 999` — LendingClub's placeholder sentinel, not a real ratio.
- `dti < 0` — physically impossible ratio (data-entry/parsing error).
- `revol_util > 100` — utilization can't legitimately exceed 100%.
- `annual_inc <= 0` — self-reported income can't be zero or negative.
- `annual_inc > $50,000,000` — two isolated outlier rows, 10x+ beyond the
  next-highest value and lacking any corroborating signal (small
  `loan_amnt`, near-zero `dti`) of a genuinely high-earning borrower.

Notebook 02's EDA of the 7 bureau features didn't flag any analogous
placeholder/garbage pattern for them (their value distributions were clean
low-cardinality counts) — noting that explicitly rather than silently leaving
`GarbageValueCleaner`'s scope unchanged with no explanation.

In [6]:
print('dti == 999:        ', (X['dti'] == 999).sum())
print('dti < 0:           ', (X['dti'] < 0).sum())
print('revol_util > 100:  ', (X['revol_util'] > 100).sum())
print('annual_inc <= 0:   ', (X['annual_inc'] <= 0).sum())
print('annual_inc > $50M: ', (X['annual_inc'] > 50_000_000).sum())

dti == 999:         38
dti < 0:            2
revol_util > 100:   4676
annual_inc <= 0:    361
annual_inc > $50M:  0


Built as a scikit-learn transformer (`src/preprocessing.py`, imported above)
so training and Phase 3 inference run identical cleaning logic — and so
notebook 04 can load the saved pipeline in a fresh kernel without redefining
the class inline.


## 4. Time-based train/test split

Same reasoning as before: lending is time-ordered, so a random shuffle-split
would leak future information into training. Splitting on `issue_d` makes the
test set genuinely simulate "loans this model hasn't seen yet."

**Caveat** (per notebook 01's censoring finding): the 2017+ test set only
contains loans that had already resolved by end of 2018 — bad loans resolve
faster than good ones pay off, so this slice likely skews toward faster,
probably worse outcomes. An evaluation-set limitation, not a modeling error.

In [7]:
TRAIN_CUTOFF = '2017-01-01'

train_df = df[df['issue_d'] < TRAIN_CUTOFF]
test_df = df[df['issue_d'] >= TRAIN_CUTOFF]

X_train, y_train = train_df[numeric_features + categorical_features], train_df['target']
X_test, y_test = test_df[numeric_features + categorical_features], test_df['target']

print(f"Train: {len(train_df):,} loans, {train_df['issue_d'].min().date()} to "
      f"{train_df['issue_d'].max().date()}, default rate {y_train.mean():.2%}")
print(f"Test:  {len(test_df):,} loans, {test_df['issue_d'].min().date()} to "
      f"{test_df['issue_d'].max().date()}, default rate {y_test.mean():.2%}")

Train: 1,026,558 loans, 2013-01-01 to 2016-12-01, default rate 20.09%
Test:  225,639 loans, 2017-01-01 to 2018-12-01, default rate 21.29%


### Rolling time-based CV folds (for regularization tuning in section 6)

Three expanding-window folds, each validating on one full year, used to pick
ElasticNet's `l1_ratio`/`C`:

| fold | train | validate |
|---|---|---|
| 1 | < 2015-01-01 | 2015 |
| 2 | < 2016-01-01 | 2016 |
| 3 | < 2017-01-01 | 2017 |

Built as a manual list of `(train_idx, val_idx)` position arrays rather than
sklearn's `PredefinedSplit` — `PredefinedSplit` assigns each row to exactly
one fold, but early rows here (2013-2014) belong to *every* fold's training
window, which `PredefinedSplit` can't express. A plain `KFold` would ignore
chronology entirely, the same problem the outer split above avoids.

**Caveat:** fold 3's validation window (2017) overlaps with the first year of
the final held-out test set above. The model never *trains* on 2017 in any
fold, but hyperparameter selection does get a peek at 2017 labels through
this fold's validation score — a mild, common compromise in rolling-origin
CV. If that's a concern, drop fold 3 and tune on folds 1-2 only.

In [8]:
def build_folds(pool_df, fold_bounds):
    """Position-array (train_idx, val_idx) pairs for each (val_start, val_end) window."""
    folds = []
    for val_start, val_end in fold_bounds:
        train_idx = np.where(pool_df['issue_d'] < val_start)[0]
        val_idx = np.where((pool_df['issue_d'] >= val_start) & (pool_df['issue_d'] < val_end))[0]
        folds.append((train_idx, val_idx))
    return folds

fold_bounds = [
    ('2015-01-01', '2016-01-01'),
    ('2016-01-01', '2017-01-01'),
    ('2017-01-01', '2018-01-01'),
]
cv_pool_df = df[df['issue_d'] < '2018-01-01'].reset_index(drop=True)
X_cv = cv_pool_df[numeric_features + categorical_features]
y_cv = cv_pool_df['target']

cv_folds = build_folds(cv_pool_df, fold_bounds)
for i, (train_idx, val_idx) in enumerate(cv_folds, 1):
    print(f"fold {i}: train n={len(train_idx):,}, val n={len(val_idx):,}")

fold 1: train n=357,907, val n=375,546
fold 2: train n=733,453, val n=293,105
fold 3: train n=1,026,558, val n=169,321


## 5. Build preprocessing pipelines

Same structure as before: `GarbageValueCleaner` → `SimpleImputer(median,
add_indicator=True)` → `StandardScaler` for logistic regression's numeric
branch (LightGBM skips scaling — tree splits are invariant to it);
`OneHotEncoder` for logistic regression's categoricals, native pandas
`category` dtype for LightGBM (via `CastToCategory`, imported from
`src/preprocessing.py` alongside `GarbageValueCleaner`).


In [9]:
# ---- Logistic regression pipeline: scaled numeric + one-hot categorical ----
lr_numeric = Pipeline([
    ('clean', GarbageValueCleaner()),
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
    ('scale', StandardScaler()),
])
lr_categorical = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
lr_preprocessor = ColumnTransformer([
    ('num', lr_numeric, numeric_features),
    ('cat', lr_categorical, categorical_features),
], verbose_feature_names_out=False)

# l1_ratio/C are placeholders here -- tuned via GridSearchCV in section 6.
# max_iter/tol kept modest for the grid-search stage (36 fits); the final
# refit in section 6 uses a more patient setting since it's a single fit.
lr_pipeline = Pipeline([
    ('preprocess', lr_preprocessor),
    ('model', LogisticRegression(solver='saga', l1_ratio=0.5, C=1.0,
                                  max_iter=300, tol=1e-3, random_state=RANDOM_STATE)),
])

In [10]:
# ---- LightGBM pipeline: unscaled numeric + native categorical dtype ----
lgbm_numeric = Pipeline([
    ('clean', GarbageValueCleaner()),
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
])
lgbm_categorical = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
])
lgbm_preprocessor = ColumnTransformer([
    ('num', lgbm_numeric, numeric_features),
    ('cat', lgbm_categorical, categorical_features),
], verbose_feature_names_out=False).set_output(transform='pandas')

lgbm_pipeline = Pipeline([
    ('preprocess', lgbm_preprocessor),
    ('cast_categorical', CastToCategory(categorical_features)),
    ('model', LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        importance_type='gain', random_state=RANDOM_STATE, verbose=-1,
    )),
])

## 6. Logistic regression: regularization selection via rolling CV

Swapping the plain logistic regression for an **ElasticNet-penalized** one —
a blend of L1 (drives weak coefficients to exactly zero, i.e. automatic
feature selection) and L2 (shrinks but keeps all coefficients, handling
correlated features like `open_acc`/`total_acc` more gracefully than pure
L1). `l1_ratio` controls the L1/L2 mix, `C` controls overall regularization
strength; both are swept via grid search since neither is known ahead of
time.

Scored on the rolling folds from section 4, not a random `KFold` — same
chronology-respecting reasoning as the outer train/test split.

**Runtime note:** `saga` on the full multi-hundred-thousand-row folds is slow
across 12 param combinations × 3 folds. Grid search runs on a 150,000-row
random subsample of the CV pool instead (folds rebuilt on the same date
boundaries) — enough to compare hyperparameters reliably, without the cost of
36 full-size fits. The **final** model below is still fit on the complete
pre-2017 training set; only the *search* is subsampled.

In [11]:
CV_SAMPLE_SIZE = 150_000
lr_cv_pool_df = cv_pool_df
if len(cv_pool_df) > CV_SAMPLE_SIZE:
    lr_cv_pool_df = cv_pool_df.sample(n=CV_SAMPLE_SIZE, random_state=RANDOM_STATE)
    lr_cv_pool_df = lr_cv_pool_df.sort_values('issue_d').reset_index(drop=True)

lr_cv_folds = build_folds(lr_cv_pool_df, fold_bounds)
X_cv_lr = lr_cv_pool_df[numeric_features + categorical_features]
y_cv_lr = lr_cv_pool_df['target']
for i, (train_idx, val_idx) in enumerate(lr_cv_folds, 1):
    print(f"fold {i}: train n={len(train_idx):,}, val n={len(val_idx):,}")

fold 1: train n=44,638, val n=46,873
fold 2: train n=91,511, val n=37,125
fold 3: train n=128,636, val n=21,364


In [12]:
param_grid = {
    'model__l1_ratio': [0.1, 0.5, 0.9],
    'model__C': [0.01, 0.1, 1, 10],
}
grid_search = GridSearchCV(
    lr_pipeline, param_grid, cv=lr_cv_folds, scoring='roc_auc', n_jobs=-1,
)
grid_search.fit(X_cv_lr, y_cv_lr)

print('Best params:', grid_search.best_params_)
print(f'Best mean CV AUC: {grid_search.best_score_:.4f}')

Best params: {'model__C': 10, 'model__l1_ratio': 0.1}
Best mean CV AUC: 0.6975


### What got zeroed out

Refitting the tuned pipeline on the full pre-2017 training set (the actual
model notebook 04 will evaluate) and checking which coefficients L1 drove to
exactly zero. A single fit, so it can afford more iterations than the grid
search above.

In [13]:
best_params = {k.replace('model__', ''): v for k, v in grid_search.best_params_.items()}

lr_pipeline = Pipeline([
    ('preprocess', lr_preprocessor),
    ('model', LogisticRegression(solver='saga', max_iter=1000, tol=1e-3,
                                  random_state=RANDOM_STATE, **best_params)),
])
lr_pipeline.fit(X_train, y_train)

lr_feature_names = lr_pipeline.named_steps['preprocess'].get_feature_names_out()
lr_coefs = lr_pipeline.named_steps['model'].coef_[0]
zeroed = lr_feature_names[lr_coefs == 0]

print(f'{len(zeroed)} / {len(lr_coefs)} coefficients set to exactly zero')
print(list(zeroed))

0 / 44 coefficients set to exactly zero
[]


**Result:** grid search picked `C=10, l1_ratio=0.1` (best mean CV AUC 0.698)
— weak overall regularization, mostly L2. At that setting, **0 of 44**
coefficients were driven to exactly zero, including `open_acc` (coef=0.10)
and `total_acc` (coef=-0.11) — both kept, and their opposite signs are a
classic sign of the multicollinearity notebook 02 flagged (r=0.70): L2 splits
their shared signal across both rather than picking one. L1 barely engaged
here — the rolling-CV AUC surface preferred flexibility over sparsity, so
this search didn't produce the automatic feature selection L1 is often used
for. That's a legitimate result, not a failed search; a follow-up grid biased
toward smaller `C` would show what a sparser model drops.


## 7. Fit final models

The tuned logistic regression pipeline was already fit above (needed for the
coefficient check). Fitting LightGBM the same way — same structure as
before, no additional tuning needed for the final fit.

In [14]:
lgbm_pipeline.fit(X_train, y_train)
print('LightGBM fit.')

LightGBM fit.


### Rolling-fold performance for LightGBM (artifact for notebook 04)

Notebook 04's stability check reports LightGBM's AUC/KS across the same 3
rolling folds, but that notebook doesn't retrain anything — so computing it
once here and saving the result, rather than duplicating this fit logic
there.

In [15]:
def ks_statistic(y_true, y_prob):
    """Max |CDF_bad(p) - CDF_good(p)| -- the textbook credit-risk KS statistic."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bad_scores = np.sort(y_prob[y_true == 1])
    good_scores = np.sort(y_prob[y_true == 0])
    thresholds = np.sort(np.unique(y_prob))
    cum_bad = np.searchsorted(bad_scores, thresholds, side='right') / len(bad_scores)
    cum_good = np.searchsorted(good_scores, thresholds, side='right') / len(good_scores)
    return np.max(np.abs(cum_bad - cum_good))

rolling_results = []
for i, (train_idx, val_idx) in enumerate(cv_folds, 1):
    fold_model = clone(lgbm_pipeline)
    fold_model.fit(X_cv.iloc[train_idx], y_cv.iloc[train_idx])
    val_prob = fold_model.predict_proba(X_cv.iloc[val_idx])[:, 1]
    val_true = y_cv.iloc[val_idx]
    rolling_results.append({
        'fold': i,
        'val_start': fold_bounds[i - 1][0],
        'val_end': fold_bounds[i - 1][1],
        'n_val': len(val_idx),
        'AUC': roc_auc_score(val_true, val_prob),
        'KS': ks_statistic(val_true, val_prob),
    })

rolling_results_df = pd.DataFrame(rolling_results)
rolling_results_df

,fold,val_start,val_end,n_val,AUC,KS
0,1,2015-01-01,2016-01-01,375546,0.72,0.32
1,2,2016-01-01,2017-01-01,293105,0.70,0.29
2,3,2017-01-01,2018-01-01,169321,0.70,0.29


## 8. Persist artifacts for notebook 04 and Phase 3

Saving the held-out test set as a parquet file rather than having notebook
04 re-derive it from the raw CSV — notebook 04's job is evaluation only, and
re-running this notebook's loading/cleaning/splitting logic there would
duplicate it and risk drifting out of sync. The saved test set includes
`int_rate` and `recoveries` alongside the model features — not used by
either model, but needed for notebook 04's business-cost framing.

In [16]:
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(lgbm_pipeline, MODELS_DIR / 'lgbm_default_risk_pipeline.joblib')
joblib.dump(lr_pipeline, MODELS_DIR / 'lr_default_risk_pipeline.joblib')
print('Saved both pipelines.')

Saved both pipelines.


In [17]:
test_artifact = test_df[numeric_features + categorical_features + ['target', 'issue_d', 'int_rate', 'recoveries']].copy()
test_artifact.to_parquet(MODELS_DIR / 'test_df.parquet', index=False)
print(f'Saved test set: {test_artifact.shape}')

Saved test set: (225639, 24)


In [18]:
pd.DataFrame(grid_search.cv_results_).to_csv(MODELS_DIR / 'lr_gridsearch_cv_results.csv', index=False)
rolling_results_df.to_csv(MODELS_DIR / 'lgbm_rolling_cv_results.csv', index=False)
print('Saved grid search results and rolling-fold results.')

Saved grid search results and rolling-fold results.


## 9. Notebook summary

- **Feature set:** 16 numeric + 4 categorical (section 2).
- **Garbage-value cleaning:** `GarbageValueCleaner` now applies five rules
  (section 3) — `dti == 999`, `dti < 0`, `revol_util > 100`,
  `annual_inc <= 0`, `annual_inc > $50M` — the last two added per notebook
  01's audit. Measured effect on this population is small: 2 rows had
  `dti < 0` fixed; the two `annual_inc > $50M` outlier rows found in
  notebook 01 aren't in this notebook's resolved population (still
  `Current` as of the data snapshot), so the new income rule affects 0 rows
  here but remains in place for Phase 3 inference-time robustness.
- **ElasticNet tuning:** rolling-CV grid search selected `C=10, l1_ratio=0.1`
  (best mean CV AUC 0.698) — weak, mostly-L2 regularization. 0 of 44
  coefficients were driven to exactly zero at that setting, including
  `open_acc`/`total_acc` (both kept, opposite-signed) — L1 didn't meaningfully
  engage; validation AUC preferred flexibility over sparsity.
- **Artifacts saved to `../models/`:** `lgbm_default_risk_pipeline.joblib`
  and `lr_default_risk_pipeline.joblib` (both fit on the full pre-2017
  training set, 1,026,558 loans), `test_df.parquet` (the held-out 2017+ test
  set, 225,639 loans, plus `int_rate`/`recoveries` for notebook 04's
  business-cost framing), `lr_gridsearch_cv_results.csv`, and
  `lgbm_rolling_cv_results.csv` (LightGBM AUC 0.70-0.72, KS 0.29-0.32 across
  the 3 rolling folds) — all ready for `04_model_evaluation.ipynb`, which
  evaluates without retraining anything.